### 2.1 Geographic Patterns: Area vs Financial Variables

We investigate whether the geographic area (Nord, Centro, Sud/Isole) is associated with differences in wealth, income, job composition, and debt levels. This is relevant because Italian macro-regions have well-known socio-economic disparities.

In [1]:
# ── Area vs Wealth, Income, Job, Debt ─────────────────────────────────────────
area_map = {1: 'Nord', 2: 'Centro', 3: 'Sud/Isole'}
job_map  = {1: 'Unemployed', 2: 'Employee', 3: 'Manager', 4: 'Entrepreneur', 5: 'Retired'}
df['Area_L'] = df['Area'].map(area_map)
df['Job_L']  = df['Job'].map(job_map)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Geographic Area vs Financial Variables', fontsize=16, fontweight='bold')

# Area vs Wealth — boxplot
sns.boxplot(data=df, x='Area_L', y='Wealth', hue='Area_L', ax=axes[0,0],
            palette='Blues', order=['Nord','Centro','Sud/Isole'], legend=False)
axes[0,0].set_title('Area vs Wealth')
axes[0,0].set_xlabel('')
axes[0,0].set_ylabel('Wealth (percentile)')

# Area vs Income — boxplot
sns.boxplot(data=df, x='Area_L', y='Income', hue='Area_L', ax=axes[0,1],
            palette='Oranges', order=['Nord','Centro','Sud/Isole'], legend=False)
axes[0,1].set_title('Area vs Income')
axes[0,1].set_xlabel('')
axes[0,1].set_ylabel('Income (percentile)')

# Area vs Job — stacked bar (proportions)
job_area_ct = pd.crosstab(df['Area_L'], df['Job_L'], normalize='index')
job_area_ct = job_area_ct[['Unemployed','Employee','Manager','Entrepreneur','Retired']]
job_area_ct.loc[['Nord','Centro','Sud/Isole']].plot(
    kind='bar', stacked=True, ax=axes[1,0], colormap='tab10', edgecolor='white', linewidth=0.5)
axes[1,0].set_title('Area vs Job Distribution')
axes[1,0].set_xlabel('')
axes[1,0].set_ylabel('Proportion')
axes[1,0].legend(fontsize=8, loc='upper right')
axes[1,0].tick_params(axis='x', rotation=0)

# Area vs Debt — boxplot
sns.boxplot(data=df, x='Area_L', y='Debt', hue='Area_L', ax=axes[1,1],
            palette='Reds', order=['Nord','Centro','Sud/Isole'], legend=False)
axes[1,1].set_title('Area vs Debt')
axes[1,1].set_xlabel('')
axes[1,1].set_ylabel('Debt (percentile)')

plt.tight_layout()
plt.show()

print('--- Median values by Area ---')
print(df.groupby('Area_L')[['Income','Wealth','Debt']].median().round(3)
      .loc[['Nord','Centro','Sud/Isole']].to_string())

NameError: name 'df' is not defined

**Interpretation — Area vs Financial Variables:**

- **Income and Wealth** show mild geographic disparities: Nord has slightly higher medians, but the distributions overlap substantially. This reflects Italian macro-economic structure but the differences are modest — geographic area alone is a poor predictor of financial status.
- **Job composition** varies: Sud/Isole has a larger share of retired clients and relatively more employees, while Nord has more diverse job profiles. This is consistent with the age structure and labor market differences across Italian regions.
- **Debt** is surprisingly uniform across regions — the median and interquartile range are similar everywhere. This suggests that debt levels are more driven by personal/family factors than by geography.

*Takeaway for clustering*: Area alone won't be a strong cluster driver, but it adds complementary information when combined with other features.

### 2.2 Investment Behavior: Investments vs Financial Education & Digital Propensity

Are clients who invest (especially in capital accumulation plans) more financially educated and more digitally active? This relationship is central to understanding the drivers of investment behavior.

In [ ]:
# ── Investments vs FinEdu and Digital ─────────────────────────────────────────
inv_map = {1: 'No investments', 2: 'Lump Sum', 3: 'Capital Acc. (PAC)'}
df['Inv_L'] = df['Investments'].map(inv_map)
inv_order = ['No investments', 'Lump Sum', 'Capital Acc. (PAC)']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Investment Type vs Financial Sophistication', fontsize=16, fontweight='bold')

# Investments vs FinEdu — violin plot (shows full distribution shape)
sns.violinplot(data=df, x='Inv_L', y='FinEdu', hue='Inv_L', ax=axes[0],
               palette=['#E8A87C','#D5CABD','#41B3A3'], order=inv_order,
               inner='quartile', cut=0, legend=False)
axes[0].set_title('Investments vs Financial Education')
axes[0].set_xlabel('')
axes[0].set_ylabel('Financial Education (percentile)')

# Investments vs Digital — violin plot
sns.violinplot(data=df, x='Inv_L', y='Digital', hue='Inv_L', ax=axes[1],
               palette=['#E8A87C','#D5CABD','#41B3A3'], order=inv_order,
               inner='quartile', cut=0, legend=False)
axes[1].set_title('Investments vs Digital Propensity')
axes[1].set_xlabel('')
axes[1].set_ylabel('Digital (percentile)')

plt.tight_layout()
plt.show()

print('--- Mean FinEdu & Digital by Investment Type ---')
print(df.groupby('Inv_L')[['FinEdu','Digital']].mean().round(3).loc[inv_order].to_string())

**Interpretation — Investments vs FinEdu & Digital:**

There is a clear monotonic gradient: clients investing in PAC (capital accumulation) plans have the highest financial education (0.538 vs 0.469 for non-investors) and digital propensity (0.555 vs 0.500). The violin shapes show that PAC investors are more concentrated in the upper range, while non-investors have a wider, flatter distribution skewed lower.

This makes intuitive sense: PAC plans require understanding of dollar-cost averaging, long-term investment horizons, and often digital platforms to manage regular payments. Lump-sum investors sit in between — they have enough financial knowledge to invest, but perhaps prefer a simpler, one-time approach.

*Takeaway*: Financial education and digital propensity are key drivers (or at least strong correlates) of investment sophistication. These variables will likely differentiate clusters.

### 2.3 Investment Behavior: Investments vs Age & Job

Do older or retired clients prefer different investment types? Does professional status influence investment choice?

**Important**: the heatmap is now normalized **by Job** (column-wise), so we read it as: *"Of all Managers, what % invests in PAC?"*. This is much more informative than normalizing by investment type, because job categories have very unequal sizes (3,284 employees vs only 107 entrepreneurs).

In [ ]:
# ── Investments vs Age and Job ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Investment Type vs Demographics', fontsize=16, fontweight='bold')

# Investments vs Age — boxplot
sns.boxplot(data=df, x='Inv_L', y='Age', hue='Inv_L', ax=axes[0],
            palette=['#4C72B0','#DD8452','#55A868'], order=inv_order, legend=False)
axes[0].set_title('Investment Type vs Age')
axes[0].set_xlabel('')
axes[0].set_ylabel('Age (years)')

# Investments vs Job — heatmap normalized BY JOB (column-wise)
# Reads: "Of all people with Job=X, what % invests in each type?"
inv_job_ct = pd.crosstab(df['Inv_L'], df['Job_L'], normalize='columns') * 100
job_order = ['Unemployed','Employee','Manager','Entrepreneur','Retired']
inv_job_ct = inv_job_ct.loc[inv_order, job_order]

sns.heatmap(inv_job_ct, annot=True, fmt='.1f', cmap='YlGnBu', ax=axes[1],
            linewidths=0.5, cbar_kws={'label': '% within each Job category'})
axes[1].set_title('Investment Type vs Job (column %, normalized by Job)')
axes[1].set_xlabel('')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print('--- Median Age by Investment Type ---')
print(df.groupby('Inv_L')['Age'].agg(['median','mean']).round(1).loc[inv_order].to_string())
print('\n--- Job category sizes (for context) ---')
print(df['Job_L'].value_counts().loc[job_order].to_string())

**Interpretation — Investments vs Age & Job:**

**Age boxplot**: Non-investors are older (median ~66y) while PAC investors are younger (median ~57y). This reflects the financial lifecycle: younger working-age clients are in the accumulation phase and benefit from PAC (regular monthly investing), while older clients either already accumulated assets (lump sum) or don't invest at all.

**Heatmap (normalized by Job column)**: The heatmap now answers: *"Of all Managers, what percentage invests in PAC?"*
- **Managers** are the most investment-active: 61.1% use PAC plans — the highest rate of any profession. Only 18.1% don't invest.
- **Employees** are also PAC-oriented (50.4%), consistent with their salary-based income being suitable for regular monthly contributions.
- **Retired** clients overwhelmingly prefer Lump Sum (45%) — they have accumulated savings and invest them in bulk. Only 22.5% use PAC.
- **Unemployed and Entrepreneurs** show higher rates of non-investment (~31-33%), likely due to income instability.

*Takeaway*: Investment type is a strong signal for both age and profession. The clustering should naturally separate active PAC investors (younger, employed/managers) from conservative lump-sum investors (older, retired).

### 2.4 Investment Behavior: Investments vs Wealth & Saving Propensity

We plot the three investment types in **separate panels** in the Wealth-Saving space. This avoids the clutter of a single overlapping scatter and lets us compare the shape, center, and dispersion of each group independently.

In [ ]:
# ── Investments vs Wealth and Saving — separate panels ─────────────────────────
inv_colors = {'No investments': '#E07A5F', 'Lump Sum': '#81B29A', 'Capital Acc. (PAC)': '#3D405B'}

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Wealth vs Saving — by Investment Type', fontsize=16, fontweight='bold')

for idx, inv_type in enumerate(inv_order):
    mask = df['Inv_L'] == inv_type
    sub = df[mask]
    axes[idx].scatter(sub['Wealth'], sub['Saving'],
                      alpha=0.2, s=10, color=inv_colors[inv_type])
    axes[idx].set_xlabel('Wealth (percentile)')
    axes[idx].set_ylabel('Saving (percentile)' if idx == 0 else '')
    axes[idx].set_title(f'{inv_type}\n(n={mask.sum()}, r={sub["Wealth"].corr(sub["Saving"]):.2f})')
    axes[idx].set_xlim(-0.05, 1.05)
    axes[idx].set_ylim(-0.05, 1.05)
    axes[idx].grid(True, alpha=0.2)
    # Add mean marker
    axes[idx].scatter(sub['Wealth'].mean(), sub['Saving'].mean(),
                      s=200, marker='X', color='red', edgecolors='black',
                      linewidths=1.5, zorder=5,
                      label=f'Mean ({sub["Wealth"].mean():.2f}, {sub["Saving"].mean():.2f})')
    axes[idx].legend(fontsize=8, loc='upper left')

plt.tight_layout()
plt.show()

print('--- Mean Wealth & Saving by Investment Type ---')
print(df.groupby('Inv_L')[['Wealth','Saving']].mean().round(3).loc[inv_order].to_string())

**Interpretation — Wealth vs Saving by Investment Type (separate panels):**

Splitting into three panels reveals what was hidden in the overlapping scatter:

- **No investments** (left): the cloud is centered lower-left (mean Wealth=0.54, Saving=0.49), with a moderate positive Wealth-Saving correlation (r=0.36). These clients save and earn less — they are the least financially active segment.
- **Lump Sum** (center): the cloud shifts upward and rightward (mean Wealth=0.58, Saving=0.49), with the strongest correlation (r=0.45). The stronger correlation suggests that lump-sum investors are more 'rational' — their saving and wealth move together. However, Saving is similar to non-investors: lump-sum investors are wealthier but not necessarily bigger savers.
- **Capital Acc. (PAC)** (right): highest mean on both axes (Wealth=0.61, Saving=0.56), but the **weakest** correlation (r=0.22). This is the key insight — PAC investors save more **regardless of their wealth level**. The weaker correlation means even PAC clients with moderate wealth tend to save actively, which makes sense: PAC is designed for systematic accumulation, attracting disciplined savers at every income level.

*Takeaway*: PAC investors save more at every wealth level; lump-sum investors are wealthier but not better savers; non-investors are behind on both dimensions.

### 2.5 Family Dynamics: FamilySize vs Debt & Saving

Larger families might carry more debt — but do they save less? The data tells a more nuanced story.

In [2]:
# ── FamilySize vs Debt and Saving ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Family Size vs Financial Behavior', fontsize=16, fontweight='bold')

# FamilySize vs Debt — boxplot
sns.boxplot(data=df, x='FamilySize', y='Debt', hue='FamilySize', ax=axes[0],
            palette='rocket_r', legend=False)
axes[0].set_title('Family Size vs Debt')
axes[0].set_xlabel('Family Size (members)')
axes[0].set_ylabel('Debt (percentile)')

# FamilySize vs Saving — boxplot
sns.boxplot(data=df, x='FamilySize', y='Saving', hue='FamilySize', ax=axes[1],
            palette='crest', legend=False)
axes[1].set_title('Family Size vs Saving Propensity')
axes[1].set_xlabel('Family Size (members)')
axes[1].set_ylabel('Saving (percentile)')

plt.tight_layout()
plt.show()

# Detailed table
fam_stats = df.groupby('FamilySize').agg(
    n_clients=('Debt', 'count'),
    avg_income=('Income', 'mean'),
    avg_debt=('Debt', 'mean'),
    avg_saving=('Saving', 'mean'),
    pct_pac=('Investments', lambda x: (x == 3).mean() * 100)
).round(3)
print('--- Financial profile by Family Size ---')
print(fam_stats.to_string())

NameError: name 'plt' is not defined

**Interpretation — FamilySize vs Debt & Saving:**

The data reveals a seemingly paradoxical pattern: **larger families (5-6 members) both carry more debt AND save more**, roughly at the same income level.

This is not actually contradictory — it reflects the **economics of large families**:

1. **Higher debt is straightforward**: more family members means more expenses (housing, education, daily costs), leading to more borrowing. Average debt rises from 0.41 (single) to 0.49 (family of 6).

2. **Higher saving is the interesting part**. The likely explanation is **precautionary saving**: large families face greater financial risk (more dependents, higher expenses), so they are *more motivated* to build a safety buffer. They may also have stronger financial discipline driven by necessity — when you have 5-6 people depending on your income, you *must* plan ahead.

3. The correlation between FamilySize and Saving is very weak (r=0.037), meaning this is a mild effect, not a dominant one. The boxplots confirm that distributions overlap substantially.

4. An alternative explanation: large families in this dataset may slightly over-represent specific demographics (e.g. certain geographic areas or job types) that happen to save more. But the effect persists at roughly constant income (~0.56-0.59), so it's not purely an income confound.

*Takeaway*: FamilySize contributes mild but real information to financial behavior — it's not redundant with income or debt alone.

### 2.6 ESG Propensity: ESG vs Age & Wealth

ESG (Environmental, Social, Governance) awareness is a growing dimension in financial services. We use hexbin plots (density-colored scatter) to see where the mass of clients concentrates.

In [ ]:
# ── ESG vs Age and Wealth ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
fig.suptitle('ESG Propensity vs Age & Wealth', fontsize=16, fontweight='bold')

# ESG vs Age — hexbin density plot
hb1 = axes[0].hexbin(df['Age'], df['ESG'], gridsize=25, cmap='YlOrRd', mincnt=1)
axes[0].set_xlabel('Age (years)')
axes[0].set_ylabel('ESG Propensity (percentile)')
axes[0].set_title('ESG vs Age')
plt.colorbar(hb1, ax=axes[0], label='Count')

# ESG vs Wealth — hexbin density plot
hb2 = axes[1].hexbin(df['Wealth'], df['ESG'], gridsize=25, cmap='YlGnBu', mincnt=1)
axes[1].set_xlabel('Wealth (percentile)')
axes[1].set_ylabel('ESG Propensity (percentile)')
axes[1].set_title('ESG vs Wealth')
plt.colorbar(hb2, ax=axes[1], label='Count')

plt.tight_layout()
plt.show()

# Detailed breakdown
print('--- Correlations with ESG ---')
print(f"  ESG vs Age:    r = {df['ESG'].corr(df['Age']):.3f}")
print(f"  ESG vs Wealth: r = {df['ESG'].corr(df['Wealth']):.3f}")
print(f"  ESG vs Income: r = {df['ESG'].corr(df['Income']):.3f}")
print(f"  ESG vs FinEdu: r = {df['ESG'].corr(df['FinEdu']):.3f}")

# ESG by age bins
df['AgeBin'] = pd.cut(df['Age'], bins=[18,30,40,50,60,70,80,96],
                       labels=['19-30','31-40','41-50','51-60','61-70','71-80','81-95'])
print('\n--- Mean ESG by Age Group ---')
print(df.groupby('AgeBin')['ESG'].agg(['mean','count']).round(3).to_string())
df = df.drop(columns=['AgeBin'])

**Interpretation — ESG vs Age & Wealth:**

**Left panel (ESG vs Age):** The hexbin plot shows client density in Age-ESG space. The darkest cells (most clients) sit around age 60-80 and ESG 0.5-0.7, but this simply reflects where most clients are demographically. The critical insight is that ESG propensity is **essentially flat across age** (r = −0.016, practically zero). If anything, the youngest clients (19-30) have the **lowest** mean ESG (0.549), while the 31-60 cohort peaks at ~0.61.

This might seem counterintuitive — aren't younger generations more ESG-conscious? In the general population, perhaps. But in a **bank client dataset**, ESG propensity is a *financial attitude measure*, not an opinion survey — it reflects whether a client actively considers ESG criteria in investment decisions. That requires a minimum level of financial sophistication that younger, less wealthy clients may not yet have developed. Young clients here are less likely to invest at all (see Section 2.3), so they have fewer opportunities to express ESG preferences through actual financial behavior.

**Right panel (ESG vs Wealth):** The pattern is clearer: the dense core of the hexbin cloud tilts upward as wealth increases (r = 0.148). The correlation extends to related variables: ESG correlates with Income (r=0.16), FinEdu (r=0.18), and Digital (r=0.15). **ESG investing behaves as a 'luxury good'**: when clients have enough wealth and financial literacy to think beyond basic returns, they can afford to incorporate sustainability preferences. Clients focused on making ends meet prioritize returns over ESG screens.

*Takeaway*: ESG propensity in this dataset is NOT driven by age, but by **financial sophistication and wealth**. It will cluster with Income, Wealth, FinEdu, and Digital — not with age. This means our affluent digital cluster (Cluster 3) will naturally show the highest ESG.

### 2.7 EDA Summary

**Key takeaways from the extended exploratory analysis:**

- **Geographic disparities** exist in income and wealth across Italian macro-regions, but are modest; debt is uniform. Area adds information but won't dominate clustering.
- **Investment behavior** is strongly linked to financial education and digital propensity — PAC investors are systematically more financially literate and digitally active.
- **Managers** are the most investment-active job category (61% use PAC); **retired clients** strongly prefer lump-sum; the **unemployed** and **entrepreneurs** show higher non-investment rates.
- **Older clients** tend toward lump-sum or no investments; PAC is more common among younger, employed clients — reflecting the financial lifecycle.
- **PAC investors** save more at every wealth level (not just because they're richer); the Wealth-Saving correlation is actually weakest for PAC users, suggesting systematic saving behavior independent of accumulated wealth.
- **Larger families** carry more debt but also save more — a precautionary saving effect driven by financial responsibility, not by income differences.
- **ESG awareness** is NOT driven by age (young clients are actually *less* ESG-aware in this dataset) but by **wealth and financial education** — it behaves as a 'luxury' financial attitude.

These patterns will resurface in the cluster profiles: the clustering algorithm should capture these natural groupings without being explicitly told about them.

In [ ]:
# ── Clean up helper columns before proceeding to preprocessing ─────────────────
helper_cols = ['Area_L', 'Job_L', 'Inv_L']
for col in helper_cols:
    if col in df.columns:
        df = df.drop(columns=[col])
print(f'Cleaned up. DataFrame shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')